# 12 RF Instrument Console v2

Stage 9 entry point for the F-engine RF instrument console. It uses the Stage 8 overlay and provides DAC tone control, ADC center/BW control, realtime scope, realtime spectrum, and shared sample0 phase metadata in one panel.


In [ ]:

from pathlib import Path
import asyncio
import sys
import time

import numpy as np
import ipywidgets as W
import plotly.graph_objects as go
from IPython.display import display

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/home/xilinx/jupyter_notebooks/t510_fengine'),
    Path('/home/xilinx/t510_fengine_bringup'),
    Path('/home/astrolab/demo-ant'),
]
PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'overlay' / 't510_fengine.bit').exists() and (root / 'python' / 't510_fengine.py').exists():
        PROJECT_ROOT = root
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('overlay/t510_fengine.bit and python/t510_fengine.py were not found')
sys.path.insert(0, str(PROJECT_ROOT))
from python.t510_fengine import T510FEngine

BITFILE = PROJECT_ROOT / 'overlay' / 't510_fengine.bit'
EXPECTED_CORE_VERSION = 0x00010006
COLORS = ['#0b5cad', '#c45200', '#217a3b', '#b3261e', '#6f42c1', '#5f6368', '#008b8b', '#9a7b00']
CHANNEL_LABELS = ['CH0 physical loopback verified'] + [f'CH{i} digital/control only' for i in range(1, 8)]
state = {
    'core': None,
    'live': False,
    'task': None,
    'last_cfg': None,
    'last_error': None,
    'last_capture_s': None,
}

def visible_mask():
    return (1 << int(visible_channels.value)) - 1

def amplitude_code():
    return int(round(float(dac_amplitude_percent.value) / 100.0 * 8192.0))

def core():
    if state['core'] is None:
        raise RuntimeError('Load overlay first')
    return state['core']

def set_status(html, *, kind='info'):
    colors = {'info': '#174ea6', 'ok': '#137333', 'warn': '#b06000', 'err': '#b3261e'}
    status_html.value = f"<pre style='margin:0;color:{colors.get(kind, '#202124')};white-space:pre-wrap'>{html}</pre>"

# Controls
download_bitstream = W.Checkbox(value=True, description='Download bitstream')
center_mhz = W.FloatSlider(value=1500.0, min=500.0, max=3500.0, step=1.0, description='ADC center MHz', readout_format='.0f', style={'description_width': '140px'}, layout=W.Layout(width='460px'))
display_bw_mhz = W.FloatSlider(value=100.0, min=5.0, max=120.0, step=5.0, description='Display BW MHz', readout_format='.0f', style={'description_width': '140px'}, layout=W.Layout(width='460px'))
dac_tone_mhz = W.FloatSlider(value=20.0, min=0.25, max=80.0, step=0.25, description='DAC tone MHz', readout_format='.2f', style={'description_width': '140px'}, layout=W.Layout(width='460px'))
dac_amplitude_percent = W.IntSlider(value=25, min=0, max=100, step=1, description='DAC amplitude %FS', style={'description_width': '140px'}, layout=W.Layout(width='460px'))
dac_phase_deg = W.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='DAC phase deg', readout_format='.0f', style={'description_width': '140px'}, layout=W.Layout(width='460px'))
visible_channels = W.IntSlider(value=1, min=1, max=8, step=1, description='Visible channels', style={'description_width': '140px'}, layout=W.Layout(width='460px'))
phase_ref_input = W.Dropdown(options=[(f'ADC{i}', i) for i in range(8)], value=0, description='Phase ref', style={'description_width': '140px'}, layout=W.Layout(width='260px'))
samples = W.Dropdown(options=[256, 512, 1024], value=512, description='Samples', style={'description_width': '140px'}, layout=W.Layout(width='260px'))
refresh_hz = W.FloatSlider(value=5.0, min=1.0, max=10.0, step=0.5, description='Refresh Hz', readout_format='.1f', style={'description_width': '140px'}, layout=W.Layout(width='460px'))
timeout_s = W.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description='Timeout s', readout_format='.1f', style={'description_width': '140px'}, layout=W.Layout(width='460px'))

load_btn = W.Button(description='Load', icon='download', button_style='primary', layout=W.Layout(width='110px'))
init_btn = W.Button(description='Init', icon='play', button_style='success', layout=W.Layout(width='110px'))
apply_btn = W.Button(description='Apply RF', icon='check', layout=W.Layout(width='110px'))
phase_reset_btn = W.Button(description='Phase reset', icon='refresh', layout=W.Layout(width='130px'))
capture_btn = W.Button(description='Capture', icon='camera', layout=W.Layout(width='110px'))
start_btn = W.Button(description='Start Live', icon='play', button_style='success', layout=W.Layout(width='130px'))
stop_btn = W.Button(description='Stop', icon='stop', button_style='warning', layout=W.Layout(width='110px'))
status_html = W.HTML(value='')

# Figures
scope_fig = go.FigureWidget()
for ch in range(8):
    scope_fig.add_scatter(x=[], y=[], mode='lines', name=f'CH{ch} I', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.4}, visible=(ch == 0))
scope_fig.update_layout(
    title='Scope: shared sample0 time preview',
    height=360,
    margin={'l': 55, 'r': 20, 't': 42, 'b': 45},
    xaxis_title='time (us from sample0)',
    yaxis_title='ADC code',
    template='plotly_white',
    showlegend=True,
)

spectrum_fig = go.FigureWidget()
for ch in range(8):
    spectrum_fig.add_scatter(x=[], y=[], mode='lines', name=f'CH{ch}', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.4}, visible=(ch == 0))
spectrum_fig.update_layout(
    title='Spectrum: baseband peak and RF center marker',
    height=360,
    margin={'l': 55, 'r': 20, 't': 42, 'b': 45},
    xaxis_title='baseband frequency (MHz)',
    yaxis_title='power (dB)',
    template='plotly_white',
    showlegend=True,
)


def update_channel_visibility():
    mask = visible_mask()
    with scope_fig.batch_update(), spectrum_fig.batch_update():
        for ch in range(8):
            vis = bool(mask & (1 << ch))
            scope_fig.data[ch].visible = vis
            spectrum_fig.data[ch].visible = vis


def apply_rf(*, initialize=False, start=False):
    c = core()
    cfg = c.apply_rf_instrument_config(
        center_hz=float(center_mhz.value) * 1_000_000.0,
        bw_hz=float(display_bw_mhz.value) * 1_000_000.0,
        tone_hz=float(dac_tone_mhz.value) * 1_000_000.0,
        amplitude=amplitude_code(),
        phase_deg=float(dac_phase_deg.value),
        enable_mask=visible_mask(),
        adc_active_mask=0x0003,
        initialize=initialize,
        start=start,
    )
    state['last_cfg'] = cfg
    update_channel_visibility()
    return cfg


def load_overlay(_=None):
    state['core'] = T510FEngine(str(BITFILE), download=bool(download_bitstream.value))
    status = state['core'].read_status()
    version = int(status['core_version'])
    msg = f"Project: {PROJECT_ROOT}\nBitfile: {BITFILE}\nCORE_VERSION=0x{version:08x}"
    if version != EXPECTED_CORE_VERSION:
        msg += f"\nExpected CORE_VERSION=0x{EXPECTED_CORE_VERSION:08x}"
        set_status(msg, kind='warn')
    else:
        set_status(msg, kind='ok')


def init_instrument(_=None):
    try:
        cfg = apply_rf(initialize=True, start=True)
        set_status(
            f"Initialized RF instrument\n"
            f"center={cfg['center_hz'] / 1e6:.3f} MHz  bw={cfg['bw_hz'] / 1e6:.1f} MHz\n"
            f"tone={cfg['tone_hz'] / 1e6:.3f} MHz  amplitude={cfg['amplitude']} codes  phase={cfg['phase_deg']:.1f} deg\n"
            f"dac_epoch={cfg['dac_phase_epoch']}  enable_mask=0x{cfg['enable_mask']:02x}",
            kind='ok',
        )
        capture_once()
    except Exception as exc:
        state['last_error'] = exc
        set_status(f"Init failed: {type(exc).__name__}: {exc}", kind='err')


def apply_instrument(_=None):
    try:
        cfg = apply_rf(initialize=False, start=False)
        set_status(
            f"Applied RF config\n"
            f"center={cfg['center_hz'] / 1e6:.3f} MHz  display_bw={cfg['bw_hz'] / 1e6:.1f} MHz\n"
            f"tone={cfg['tone_hz'] / 1e6:.3f} MHz  amplitude={cfg['amplitude']} codes  phase={cfg['phase_deg']:.1f} deg\n"
            f"dac_epoch={cfg['dac_phase_epoch']}  enable_mask=0x{cfg['enable_mask']:02x}",
            kind='ok',
        )
        capture_once()
    except Exception as exc:
        state['last_error'] = exc
        set_status(f"Apply failed: {type(exc).__name__}: {exc}", kind='err')


def reset_phase(_=None):
    try:
        epoch = core().reset_dac_phase()
        set_status(f"DAC phase reset: epoch={epoch}", kind='ok')
        capture_once()
    except Exception as exc:
        state['last_error'] = exc
        set_status(f"Phase reset failed: {type(exc).__name__}: {exc}", kind='err')


def format_status(status, analysis, elapsed_s):
    fps = 1.0 / elapsed_s if elapsed_s > 0 else 0.0
    state['last_capture_s'] = elapsed_s
    lines = [
        f"CORE_VERSION=0x{int(status['core_version']):08x}  streaming={int(status['streaming'])}  rfdc_valid=0x{int(status['rfdc_current_valid_mask']):04x}",
        f"sample0={analysis['sample0']}  nsamp={analysis['count']}  sample_rate={analysis['sample_rate_hz'] / 1e6:.2f} MS/s  FPS={fps:.2f}",
        f"ADC center={float(center_mhz.value):.3f} MHz  display BW={float(display_bw_mhz.value):.1f} MHz  DAC tone={float(dac_tone_mhz.value):.3f} MHz",
        f"DAC amplitude={amplitude_code()} codes ({int(dac_amplitude_percent.value)}%FS)  DAC phase={float(dac_phase_deg.value):.1f} deg  phase_ref=ADC{int(phase_ref_input.value)}",
    ]
    for ch in analysis['inputs']:
        pk = analysis['peaks'][ch]
        label = CHANNEL_LABELS[ch]
        lines.append(
            f"CH{ch}: {label}  peak={float(pk['peak_mhz']):+.4f} MHz  RF={float(pk['rf_peak_mhz']):.4f} MHz  "
            f"phase={float(pk['coherent_phase_deg']):+.1f} deg  dphase={float(pk.get('delta_coherent_phase_deg', 0.0)):+.1f} deg  "
            f"SNR={float(pk['snr_db']):.1f} dB  RMS={float(pk['rms']):.1f}  max={float(pk['max_abs_code']):.0f}"
        )
    return '\n'.join(lines)


def update_figures(analysis):
    bw = float(display_bw_mhz.value)
    x_min, x_max = -bw / 2.0, bw / 2.0
    mask = visible_mask()
    with scope_fig.batch_update():
        for ch in range(8):
            if ch in analysis['scope'] and (mask & (1 << ch)):
                trace = analysis['scope'][ch]
                scope_fig.data[ch].x = trace['time_us']
                scope_fig.data[ch].y = trace['waveform']
                scope_fig.data[ch].visible = True
            else:
                scope_fig.data[ch].visible = False
        scope_fig.layout.title = f"Scope: sample0={analysis['sample0']}  input_mask=0x{analysis['input_mask']:02x}"
    with spectrum_fig.batch_update():
        for ch in range(8):
            if ch in analysis['spectrum'] and (mask & (1 << ch)):
                spec = analysis['spectrum'][ch]
                spectrum_fig.data[ch].x = spec['freq_mhz']
                spectrum_fig.data[ch].y = spec['power_db']
                spectrum_fig.data[ch].visible = True
            else:
                spectrum_fig.data[ch].visible = False
        spectrum_fig.layout.xaxis.range = [x_min, x_max]
        spectrum_fig.layout.title = f"Spectrum: center={float(center_mhz.value):.3f} MHz  BW={bw:.1f} MHz"


def capture_once(_=None):
    try:
        c = core()
        t0 = time.monotonic()
        preview = c.capture_preview_fast(n=int(samples.value), input_mask=visible_mask(), timeout=float(timeout_s.value))
        analysis = c.compute_scope_spectrum(preview, display_bw_hz=float(display_bw_mhz.value) * 1_000_000.0, phase_ref_input=int(phase_ref_input.value))
        status = c.read_status()
        elapsed = time.monotonic() - t0
        update_figures(analysis)
        set_status(format_status(status, analysis, elapsed), kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f"Capture failed: {type(exc).__name__}: {exc}", kind='err')


async def live_loop():
    while state['live']:
        t0 = time.monotonic()
        capture_once()
        elapsed = time.monotonic() - t0
        period = 1.0 / max(float(refresh_hz.value), 0.1)
        await asyncio.sleep(max(0.0, period - elapsed))


def start_live(_=None):
    if state['live']:
        return
    state['live'] = True
    state['task'] = asyncio.create_task(live_loop())
    set_status('Live started', kind='ok')


def stop_live(_=None):
    state['live'] = False
    set_status('Live stopped', kind='warn')

load_btn.on_click(load_overlay)
init_btn.on_click(init_instrument)
apply_btn.on_click(apply_instrument)
phase_reset_btn.on_click(reset_phase)
capture_btn.on_click(capture_once)
start_btn.on_click(start_live)
stop_btn.on_click(stop_live)
visible_channels.observe(lambda change: update_channel_visibility(), names='value')

control_bar = W.HBox([load_btn, init_btn, apply_btn, phase_reset_btn, capture_btn, start_btn, stop_btn])
dac_box = W.VBox([dac_tone_mhz, dac_amplitude_percent, dac_phase_deg, visible_channels])
adc_box = W.VBox([center_mhz, display_bw_mhz, W.HBox([samples, phase_ref_input]), refresh_hz, timeout_s])
controls = W.HBox([dac_box, adc_box], layout=W.Layout(gap='24px', align_items='flex-start'))
header = W.HTML("<h2 style='margin:0 0 8px 0'>T510 F-engine RF Instrument Console v2</h2>")
body = W.VBox([header, control_bar, controls, status_html, W.HBox([scope_fig, spectrum_fig])], layout=W.Layout(gap='10px'))
set_status(f"Ready. Project={PROJECT_ROOT}\nOpen overlay: {BITFILE}", kind='info')
display(body)
